# 🦅 Claw — وكيل شخصي ذكي (ثنائي اللغة)

وكيل "Claw" بأسلوب OpenClaw: شخصية، ذاكرة دائمة، وأدوات (بحث ويب، تنفيذ بايثون، وقت، ذاكرة).
يعمل على **Groq** (qwen3.8-27b) — سريع ومجاني.

**طريقة الاستخدام:**
1. شغّل الخلايا بالترتيب (Run All).
2. تظهر واجهة Gradio — اسولف مع الوكيل مباشرة.
3. أو فعّل خلية Telegram عشان تتواصل معاه عبر البوت.

> ⚠️ ملاحظة: جلسة Kaggle تنتهي بعد ~9-12 ساعة. الوكيل تفاعلي (مش 24/7 دائم).

In [ ]:
# ⚙️ التثبيت (مرة واحدة)
!pip install -q gradio ddgs requests
print("✓ dependencies ready")

In [ ]:
# 🔑 المفاتيح — تُحقن تلقائياً من GitHub Secrets وقت الرفع
import os
os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY", "__GROQ_KEY__")
os.environ.setdefault("CLAW_MODEL", "qwen/qwen3.8-27b")

os.environ.setdefault("CLAW_TG_TOKEN", "__CLAW_TG_TOKEN__")
os.environ.setdefault("CLAW_TG_ALLOW", "__CLAW_TG_ALLOW__")
print("✓ keys set (from injection)")


In [ ]:
# 🤖 نواة الوكيل
"""
🦅 CLAW — OpenClaw-style personal agent (bilingual AR/EN)
A self-contained agent with persona, memory, and tool use.
Runs on Groq (qwen3.8-27b / gpt-oss-120b). No heavy dependencies.
"""
import os
import re
import json
import time
import datetime
import requests
import subprocess

GROQ_KEY = os.environ.get("GROQ_API_KEY", "")
MODEL = os.environ.get("CLAW_MODEL", "qwen/qwen3.8-27b")
REASONING_MODEL = "openai/gpt-oss-120b"
MEMORY_FILE = os.environ.get("CLAW_MEMORY", "claw_memory.md")

API_URL = "https://api.groq.com/openai/v1/chat/completions"

SYSTEM_PROMPT = """أنت "Claw" 🦅 — وكيل ذكي شخصي، ثنائي اللغة (عربي/إنجليزي).

شخصيتك:
- مباشر، بدون مجاملات زائدة، عملي ويقدّم نتائج.
- عندك رأي خاص وتختلف بهدوء عند الحاجة.
- ترد بنفس لغة المُستخدِم (عربي أو إنجليزي).

قدراتك:
- عندك أدوات تنفّذها: بحث ويب، تنفيذ بايثون، قراءة/حفظ الذاكرة، الوقت الحالي.
- عندما تحتاج أداة، تُخرج كتلة JSON بالصيغة التالية (ولها فقط، بدون نص آخر):
{"tool":"اسم_الأداة","args":{...}}
- الأدوات المتاحة: web_search, run_python, get_time, read_memory, clear_memory
- بعد تنفيذ الأداة ستصلك النتيجة، ثم تكمل ردّك النهائي للمستخدم.

سلوكك:
- إذا أمكن إجابة السؤال مباشرة بدون أداة، أجب مباشرة.
- استخدم أدوات فقط عندما تساعد فعلاً.
- للتذكُّر عبر الجلسات: استخدم أداة save_memory لحفظ معلومات مهمة في ملف الذاكرة.
- حدّث ذاكرتك بالاسم والتفضيلات والقرارات المهمة.
"""

# ---------------- Memory ----------------
def ensure_memory():
    if not os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "w", encoding="utf-8") as f:
            f.write("# Claw Memory\n\n")

def read_memory():
    ensure_memory()
    with open(MEMORY_FILE, "r", encoding="utf-8") as f:
        return f.read()[-4000:]

def save_memory(content):
    ensure_memory()
    with open(MEMORY_FILE, "a", encoding="utf-8") as f:
        f.write("\n" + content + "\n")
    return "saved"

def clear_memory():
    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        f.write("# Claw Memory\n\n")
    return "memory cleared"

# ---------------- Tools ----------------
def tool_web_search(query):
    try:
        from ddgs import DDGS
        results = list(DDGS().text(query, max_results=5))
    except Exception:
        results = []
    if results:
        out = []
        for x in results:
            out.append(f"- {x.get('title','')}: {x.get('href','')}")
        return "\n".join(out)
    # fallback: requests-html duckduckgo
    try:
        r = requests.get("https://html.duckduckgo.com/html/", params={"q": query},
                         headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
        titles = re.findall(r'class="result__a"[^>]*>(.*?)</a>', r.text)[:5]
        clean = lambda s: re.sub(r"<[^>]+>", "", s)
        out = [f"- {clean(t)}" for t in titles]
        return "\n".join(out) if out else "no results"
    except Exception as e:
        return f"search error: {e}"

def tool_run_python(code):
    try:
        result = subprocess.run(["python3", "-c", code], capture_output=True,
                                text=True, timeout=20, cwd="/tmp")
        out = result.stdout.strip() or result.stderr.strip()
        return out[:800] or "(no output)"
    except Exception as e:
        return f"error: {e}"

def tool_get_time():
    now = datetime.datetime.now()
    return now.strftime("%Y-%m-%d %H:%M %Z") + " | " + now.strftime("%A")

TOOLS = {
    "web_search": {"fn": tool_web_search, "desc": "بحث ويب: web_search(query: نص)"},
    "run_python": {"fn": tool_run_python, "desc": "تنفيذ بايثون: run_python(code: نص الكود)"},
    "get_time":   {"fn": tool_get_time,   "desc": "الوقت والتاريخ الحالي"},
    "read_memory": {"fn": read_memory,    "desc": "قراءة ملف الذاكرة"},
    "save_memory": {"fn": save_memory,    "desc": "حفظ للذاكرة: save_memory(content: نص)"},
    "clear_memory": {"fn": clear_memory,  "desc": "مسح الذاكرة"},
}

TOOL_DESC = "\n".join(f"- {t['desc']}" for t in TOOLS.values())

def call_groq(messages, model=MODEL, max_tokens=1024, temp=0.7, retries=4):
    payload = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temp,
    }
    headers = {"Authorization": f"Bearer {GROQ_KEY}",
               "Content-Type": "application/json"}
    for attempt in range(retries):
        try:
            r = requests.post(API_URL, headers=headers, json=payload, timeout=90)
            if r.status_code == 429:
                time.sleep(3 * (attempt + 1) + 2)
                continue
            r.raise_for_status()
            return r.json()["choices"][0]["message"]["content"]
        except requests.exceptions.HTTPError as e:
            if e.response is not None and e.response.status_code == 429:
                time.sleep(3 * (attempt + 1) + 2)
                continue
            raise
        except requests.exceptions.ConnectionError:
            time.sleep(3 * (attempt + 1))
    raise RuntimeError("Groq rate-limited after retries")

def extract_tool_call(text):
    # try strict JSON block
    m = re.search(r'\{[^{}]*"tool"[^{}]*\}', text, re.S)
    if m:
        try:
            d = json.loads(m.group(0))
            if isinstance(d.get("tool"), str):
                return d["tool"], d.get("args", {})
        except Exception:
            pass
    # fallback: tool name then args
    m2 = re.search(r'tool["\s:=]+([A-Za-z_]+)', text)
    if m2:
        name = m2.group(1)
        args = {}
        for k in ("query", "code", "content"):
            mm = re.search(k + r'["\s:=]+"?([^",}\n]+)', text)
            if mm:
                args[k] = mm.group(1).strip()
        return name, args
    return None, None

def agent_run(user_input, history=None, max_steps=4):
    history = history or []
    memory = read_memory()
    sys_prompt = SYSTEM_PROMPT + f"\n\nأدواتك المتاحة:\n{TOOL_DESC}\n\nذاكرتك الحالية:\n{memory}\n"
    messages = [{"role": "system", "content": sys_prompt}]
    messages += history[-8:]
    messages.append({"role": "user", "content": user_input})

    for _ in range(max_steps):
        time.sleep(1.2)  # be gentle on rate limits
        reply = call_groq(messages)
        tool_name, args = extract_tool_call(reply)
        if tool_name and tool_name in TOOLS:
            messages.append({"role": "assistant", "content": reply})
            try:
                result = TOOLS[tool_name]["fn"](**args) if args else TOOLS[tool_name]["fn"]()
            except TypeError:
                result = TOOLS[tool_name]["fn"]()
            messages.append({"role": "user",
                             "content": f"[نتيجة أداة {tool_name}]: {result}\nأكمل ردّك النهائي."})
        else:
            return reply, history + [
                {"role": "user", "content": user_input},
                {"role": "assistant", "content": reply},
            ]
    return "انتهت الخطوات دون ردّ نهائي.", history
print('✓ claw_agent loaded')

In [ ]:
# 🎛️ واجهة Gradio — اسولف مع Claw
import gradio as gr

def respond(user_msg, chat_history):
    if chat_history is None:
        chat_history = []
    # convert gr history to openai-style
    hist = []
    for u, a in chat_history:
        if u: hist.append({"role": "user", "content": u})
        if a: hist.append({"role": "assistant", "content": a})
    reply, hist = agent_run(user_msg, hist)
    chat_history.append((user_msg, reply))
    return "", chat_history

with gr.Blocks(title="Claw 🦅", theme="soft") as demo:
    gr.Markdown("## 🦅 Claw — وكيلك الشخصي الثنائي اللغة\nأسلوب مباشر، ذاكرة دائمة، وأدوات بحث وتنفيذ.")
    chatbot = gr.Chatbot(height=480)
    msg = gr.Textbox(placeholder="اكتب رسالتك هنا... (عربي أو إنجليزي)", label="رسالتك")
    clear = gr.ClearButton([msg, chatbot])
    msg.submit(respond, [msg, chatbot], [msg, chatbot])

demo.launch(debug=False, server_name="0.0.0.0", server_port=7860)
print("✓ Gradio UI running — افتح الرابط (port 7860) من Kaggle")

In [ ]:
# 📱 وضع تيليغرام (اختياري) — تواصل مع Claw عبر البوت
# شغّل هذه الخلية بدل/بعد Gradio. يعمل طول ما الجلسة حيّة.
import requests as _rq, time as _t

TG_TOKEN = os.environ.get("CLAW_TG_TOKEN", "")
TG_ALLOW = {int(x) for x in os.environ.get("CLAW_TG_ALLOW", "").split(",") if x.strip()}
TG_API = f"https://api.telegram.org/bot{TG_TOKEN}"
hist = []  # ذاكرة المحادثة لكل جلسة

def _tg(method, **kw):
    return _rq.post(f"{TG_API}/{method}", json=kw).json()

_tg("deleteWebhook")  # تأكد ما في webhook يشغّلنا
off = 0
print("✓ Telegram bot running — اكتب للبوت @Mr65bot")
if not TG_TOKEN:
    print("⚠️ لا يوجد توكن بوت — ضع CLAW_TG_TOKEN")

try:
    while True:
        up = _tg("getUpdates", timeout=30, offset=off, allowed_updates=["message"]).get("result", [])
        for u in up:
            off = u["update_id"] + 1
            msg = u.get("message", {})
            cid = msg.get("chat", {}).get("id")
            txt = msg.get("text", "")
            if not txt or (TG_ALLOW and cid not in TG_ALLOW):
                continue
            try:
                reply, hist = agent_run(txt, hist)
            except Exception as e:
                reply = f"⚠️ خطأ: {e}"
            _tg("sendMessage", chat_id=cid, text=reply)
        _t.sleep(1)
except KeyboardInterrupt:
    print("stopped")